# Report figures

Figures for the write-up. Built from scratch 2026-07-20.

Started with the two requested first:
1. Predicted vs measured local width (dev-track LOTO)
2. Waviness (5mm-smoothed width): predicted vs measured

Remaining figures are stubbed out at the end (see TODO section) -- not built yet.

Both figures below refit the PRIMARY model (linear3 BayesianRidge, same recipe as
`scripts/run_linear_baseline.py`) per LOTO fold so we have the raw per-window
(x, y_true, mu, sigma) arrays to plot -- `results/linear_baseline/*/metrics.json`
only stores aggregate metrics, not the raw arrays.

In [ ]:
import sys
from pathlib import Path


def _find_repo_root(start=None):
    p = Path(start or Path.cwd()).resolve()
    for candidate in [p, *p.parents]:
        if (candidate / 'pyproject.toml').exists():
            return candidate
    raise RuntimeError('repo root not found (no pyproject.toml in any parent)')


REPO = _find_repo_root()
sys.path.insert(0, str(REPO / 'src'))
sys.path.insert(0, str(REPO / 'scripts'))

import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import BayesianRidge
from sklearn.preprocessing import StandardScaler

from preprocessing import DEV_TRACKS, loto_cv_splits
from thermal_features import FEATURE_NAMES

FIG_DIR = REPO / 'results' / 'report_figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
DATASET_RUN = sorted((REPO / 'processed_data' / 'datasets').iterdir())[-1].name
cache = REPO / 'processed_data' / 'features' / f'{DATASET_RUN}_thermal_v1.npz'
d = np.load(cache, allow_pickle=False)
X, y = d['features'], d['width_mean_mm']
track, valid = d['track_id'], d['valid']
x_all = d['x_mm']
assert set(np.unique(track[valid]).tolist()) == set(DEV_TRACKS)

_N = FEATURE_NAMES[:18]
_J_AREA, _J_PEAK, _J_TAIL = (_N.index('mean_area_1500'), _N.index('mean_peak'),
                             _N.index('mean_tail_len'))


def linear3(A):
    return np.column_stack([np.sqrt(A[:, _J_AREA]), A[:, _J_PEAK], A[:, _J_TAIL]])


F = linear3(X)
print(f'dataset_run={DATASET_RUN}, n_valid={valid.sum()}, dev_tracks={DEV_TRACKS}')

## 1. Predicted vs measured local width (dev-track LOTO)

Same model/standardization/inflation rule as `run_linear_baseline.py`'s `linear3`
PRIMARY variant, refit here per fold to keep the raw per-window arrays for plotting.

In [ ]:
fold_data = []
biases = {}
for fold_i, (train_tracks, val_track) in enumerate(loto_cv_splits(), start=1):
    tr = np.isin(track, train_tracks) & valid
    va = (track == val_track) & valid
    scaler = StandardScaler().fit(F[tr])
    model = BayesianRidge().fit(scaler.transform(F[tr]), y[tr])
    mu, sigma = model.predict(scaler.transform(F[va]), return_std=True)
    biases[fold_i] = float(np.median(mu - y[va]))
    fold_data.append({'fold': fold_i, 'val_track': int(val_track),
                       'x': x_all[va], 'y_true': y[va], 'mu': mu, 'sigma': sigma})

# cross-fold empirical variance inflation -- same rule as run_gp_calib_inflation.py
for f in fold_data:
    v_k = float(np.mean([b ** 2 for j, b in biases.items() if j != f['fold']]))
    f['sigma_inflated'] = np.sqrt(f['sigma'] ** 2 + v_k)

pooled_mae = np.mean(np.abs(np.concatenate([f['y_true'] for f in fold_data]) -
                             np.concatenate([f['mu'] for f in fold_data])))
print(f'pooled MAE = {pooled_mae:.4f} mm  (reference: run_linear_baseline.py reports 0.1189)')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4), sharey=True)
for ax, f in zip(axes, fold_data):
    order = np.argsort(f['x'])
    x, y_true, mu, sig = (f['x'][order], f['y_true'][order],
                          f['mu'][order], f['sigma_inflated'][order])
    ax.scatter(x, y_true, s=8, alpha=0.5, color='#333333', label='measured width')
    ax.plot(x, mu, color='#cc3333', lw=1.5, label='predicted (linear3)')
    ax.fill_between(x, mu - 2 * sig, mu + 2 * sig, color='#cc3333', alpha=0.2,
                     label='±2σ (inflated)')
    ax.set_title(f"track {f['val_track']} (held out)")
    ax.set_xlabel('x (mm)')
axes[0].set_ylabel('width (mm)')
axes[0].legend(loc='upper right', fontsize=8)
fig.suptitle('Predicted vs measured local width — LOTO dev-track validation (linear3 PRIMARY)')
fig.tight_layout()
fig.savefig(FIG_DIR / 'width_loto_comparison.png', dpi=150)
plt.show()

## 2. Waviness (5mm-smoothed width): predicted vs measured

Same 5mm centered physical-distance smoothing as `scripts/probe_waviness_signal.py`,
applied independently to the true and predicted width series from the fold data above.

In [ ]:
SMOOTH_WINDOW_MM = 5.0


def smooth_by_x(x, w, window_mm):
    out = np.empty_like(w)
    for i, xi in enumerate(x):
        m = np.abs(x - xi) <= window_mm / 2
        out[i] = w[m].mean()
    return out


for f in fold_data:
    order = np.argsort(f['x'])
    f['x_sorted'] = f['x'][order]
    f['y_true_sorted'] = f['y_true'][order]
    f['mu_sorted'] = f['mu'][order]
    f['y_true_wavy'] = smooth_by_x(f['x_sorted'], f['y_true_sorted'], SMOOTH_WINDOW_MM)
    f['mu_wavy'] = smooth_by_x(f['x_sorted'], f['mu_sorted'], SMOOTH_WINDOW_MM)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4), sharey=True)
for ax, f in zip(axes, fold_data):
    ax.plot(f['x_sorted'], f['y_true_wavy'], color='#333333', lw=2,
            label='measured (5mm smoothed)')
    ax.plot(f['x_sorted'], f['mu_wavy'], color='#cc3333', lw=2, ls='--',
            label='predicted (5mm smoothed)')
    ax.set_title(f"track {f['val_track']} (held out)")
    ax.set_xlabel('x (mm)')
axes[0].set_ylabel('width (mm), smoothed')
axes[0].legend(loc='upper right', fontsize=8)
fig.suptitle('Waviness: predicted vs measured (5mm-smoothed width) — LOTO dev-track validation')
fig.tight_layout()
fig.savefig(FIG_DIR / 'waviness_loto_comparison.png', dpi=150)
plt.show()

## 3. Track 21 final prediction

PRIMARY (linear3 BayesianRidge) predictions on track 21, cached by `run_linear_baseline.py`
(`t21_predictions_primary.npz`). SEM band width (0.379mm) shown as a reference line.

**Not yet included**: phys_gp corroboration curve -- `predict_track21.py` hasn't been
written yet, so there is no cached phys_gp track-21 prediction array, only the summary
range noted in PREPROCESSING_PLAN.md (0.37-0.46mm). Add once that script exists.

In [ ]:
t21 = np.load(REPO / 'results' / 'linear_baseline' / DATASET_RUN / 't21_predictions_primary.npz')
x21, mu21, sig21 = t21['x_mm'], t21['mu_mm'], t21['sigma_mm']
order21 = np.argsort(x21)
x21, mu21, sig21 = x21[order21], mu21[order21], sig21[order21]

SEM_BAND_T21_MM = 0.379

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(x21, mu21, color='#cc3333', lw=1.5, label='predicted width (linear3 PRIMARY)')
ax.fill_between(x21, mu21 - 2 * sig21, mu21 + 2 * sig21, color='#cc3333', alpha=0.2,
                label='±2σ (inflated)')
ax.axhline(SEM_BAND_T21_MM, color='#333333', ls=':', lw=1.5,
           label=f'SEM band width ({SEM_BAND_T21_MM} mm)')
ax.set_xlabel('x (mm)')
ax.set_ylabel('width (mm)')
ax.set_title('Track 21 (held-out test) -- final prediction, thermal input only')
ax.legend(loc='upper right', fontsize=9)
fig.tight_layout()
fig.savefig(FIG_DIR / 'track21_prediction.png', dpi=150)
plt.show()

## 4. Calibration diagnostics (PIT + coverage)

Uses the pooled LOTO fold data (with cross-fold variance inflation) from section 1.
PIT = Φ((y_true − μ)/σ); a well-calibrated model gives a flat/uniform PIT histogram.
Coverage compares the nominal 50%/90% interval to the empirical fraction of true
values actually inside it -- same definitions as `run_gp_baseline.py`'s `gaussian_metrics`.

In [ ]:
from scipy.stats import norm

y_pooled = np.concatenate([f['y_true'] for f in fold_data])
mu_pooled = np.concatenate([f['mu'] for f in fold_data])
sig_pooled = np.concatenate([f['sigma_inflated'] for f in fold_data])
z_pooled = (y_pooled - mu_pooled) / sig_pooled
pit = norm.cdf(z_pooled)

Z_50, Z_90 = norm.ppf(0.75), norm.ppf(0.95)
cov50 = float(np.mean(np.abs(z_pooled) <= Z_50))
cov90 = float(np.mean(np.abs(z_pooled) <= Z_90))
print(f'coverage_50 = {cov50:.2f} (nominal 0.50)   coverage_90 = {cov90:.2f} (nominal 0.90)')

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(pit, bins=10, range=(0, 1), color='#cc3333', edgecolor='white')
axes[0].axhline(len(pit) / 10, color='#333333', lw=1, ls=':')
axes[0].set_xlabel('PIT')
axes[0].set_ylabel('count')
axes[0].set_title('PIT histogram (dotted = uniform reference)')

nominal = [0.5, 0.9]
empirical = [cov50, cov90]
xpos = np.arange(2)
axes[1].bar(xpos - 0.15, nominal, width=0.3, color='#888888', label='nominal')
axes[1].bar(xpos + 0.15, empirical, width=0.3, color='#cc3333', label='empirical')
axes[1].set_xticks(xpos)
axes[1].set_xticklabels(['50%', '90%'])
axes[1].set_ylim(0, 1)
axes[1].set_title('Coverage: nominal vs empirical')
axes[1].legend()
fig.suptitle('Calibration — pooled LOTO dev-track validation (linear3 PRIMARY, inflated σ)')
fig.tight_layout()
fig.savefig(FIG_DIR / 'calibration_diagnostics.png', dpi=150)
plt.show()

## 5. Model comparison: MAE alone is not enough

Evaluation criterion (4) explicitly scores "calibration and usefulness of predicted
uncertainty" -- an MAE-only comparison misses this. Below: MAE, NLL, CRPS, and
coverage_90 side by side, using post-inflation numbers where the cross-fold variance
inflation was applied (`area1d`/`full18`/`linear3` via `run_linear_baseline.py`,
`phys_gp` via `run_gp_calib_inflation.py`). `anchor GP` was never inflation-corrected
-- it was already ruled out on MAE alone, so raw numbers are shown with that caveat.

Per the paper (Section 5, "Suggested Processing and Evaluation"): "negative log
likelihood **or** continuous ranked probability score" -- these are alternatives, not
both required. Both are shown here since both were already computed at no extra cost.

In [ ]:
import json

lb = json.load(open(REPO / 'results' / 'linear_baseline' / DATASET_RUN / 'metrics.json'))
gpm = json.load(open(REPO / 'results' / 'gp_physmean' / DATASET_RUN / 'metrics.json'))
gci = json.load(open(REPO / 'results' / 'gp_calib_inflation' / DATASET_RUN / 'metrics.json'))

# (mae, nll, crps, coverage_90, is_primary) -- use AFTER cross-fold inflation where available;
# anchor GP was never inflation-corrected (already ruled out on MAE alone), shown raw with a note.
anchor = gpm['variants']['anchor_ref']['pooled']['full']['gp']
phys_gp_inflated = gci['pooled']['after']

models = [
    ('constant', gpm['variants']['anchor_ref']['pooled']['full']['constant']['mae'],
     None, None, None, False),
    ('anchor GP\n(RBF, raw)', anchor['mae'], anchor['nll'], anchor['crps'],
     anchor['coverage_90'], False),
    ('area1d', lb['variants']['area1d']['pooled']['after']['mae'],
     lb['variants']['area1d']['pooled']['after']['nll'],
     lb['variants']['area1d']['pooled']['after']['crps'],
     lb['variants']['area1d']['pooled']['after']['coverage_90'], False),
    ('full18', lb['variants']['full18']['pooled']['after']['mae'],
     lb['variants']['full18']['pooled']['after']['nll'],
     lb['variants']['full18']['pooled']['after']['crps'],
     lb['variants']['full18']['pooled']['after']['coverage_90'], False),
    ('phys_gp\n(inflated)', phys_gp_inflated['mae'], phys_gp_inflated['nll'],
     phys_gp_inflated['crps'], phys_gp_inflated['coverage_90'], False),
    ('linear3\n(PRIMARY)', lb['variants']['linear3']['pooled']['after']['mae'],
     lb['variants']['linear3']['pooled']['after']['nll'],
     lb['variants']['linear3']['pooled']['after']['crps'],
     lb['variants']['linear3']['pooled']['after']['coverage_90'], True),
]

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
names = [m[0] for m in models]
colors = ['#cc3333' if m[5] else '#888888' for m in models]

maes = [m[1] for m in models]
axes[0].bar(names, maes, color=colors)
axes[0].set_ylabel('MAE (mm)')
axes[0].set_title('Point-prediction error')

nll_names = [m[0] for m in models if m[2] is not None]
nlls = [m[2] for m in models if m[2] is not None]
nll_colors = [c for m, c in zip(models, colors) if m[2] is not None]
axes[1].bar(nll_names, nlls, color=nll_colors)
axes[1].set_ylabel('NLL (lower is better)')
axes[1].set_title('Probabilistic fit: NLL')
axes[1].axhline(0, color='black', lw=0.5)

crps_names = [m[0] for m in models if m[3] is not None]
crpss = [m[3] for m in models if m[3] is not None]
crps_colors = [c for m, c in zip(models, colors) if m[3] is not None]
axes[2].bar(crps_names, crpss, color=crps_colors)
axes[2].set_ylabel('CRPS (mm, lower is better)')
axes[2].set_title('Probabilistic fit: CRPS')

cov_names = [m[0] for m in models if m[4] is not None]
covs = [m[4] for m in models if m[4] is not None]
cov_colors = [c for m, c in zip(models, colors) if m[4] is not None]
axes[3].bar(cov_names, covs, color=cov_colors)
axes[3].axhline(0.9, color='#333333', ls=':', lw=1.5, label='nominal 90%')
axes[3].set_ylabel('empirical coverage_90')
axes[3].set_title('Calibration sharpness (closer to 0.90 line = better)')
axes[3].set_ylim(0, 1.05)
axes[3].legend(fontsize=8)

for ax in axes:
    ax.tick_params(axis='x', labelsize=8)
fig.suptitle('Model comparison — MAE alone does not capture calibration quality (criterion 4)')
fig.tight_layout()
fig.savefig(FIG_DIR / 'model_comparison_full.png', dpi=150)
plt.show()

print('note: area1d/full18 have lower NLL than linear3 but overcover (0.97/0.98 vs nominal 0.90) --')
print('their intervals are wider/more conservative, not more accurate. linear3 is closest to')
print('nominal coverage (0.92) while also having the best MAE and CRPS -- the more complete')
print('argument for PRIMARY. NLL and CRPS are alternatives per the paper (Section 5), not both')
print('required -- shown together here since both were already computed at no extra cost.')

## 6. Target extraction: old vs new method (same real height-map data)

Old method (`_column_track_boundary`, per-column MAD outlier detection, pre-2026-07-16)
reproduced here from git commit `e139843` for comparison only -- it was deleted from
`src/preprocessing.py` when the NaN-valley redefinition was committed. New method is
imported directly from the current `src/preprocessing.py`. Both run on the SAME raw
height-map data for track 8, x=40-60mm, so any difference is purely the extraction
method, not the input.

In [ ]:
from nsf_fmrg_data import load_wyko_asc, robust_plane_detrend, THERMAL_MM_PER_FRAME
from preprocessing import extract_local_width_stats as new_extract_local_width_stats


# --- old (pre-2026-07-16) extraction, reproduced from commit e139843 ---
def _old_column_track_boundary(col, y_mm, mad_k=3.0, min_valid_points=5):
    finite = np.isfinite(col)
    if finite.sum() < min_valid_points:
        return None, None, False
    med = np.median(col[finite])
    mad = 1.4826 * np.median(np.abs(col[finite] - med))
    thr = mad_k * max(mad, 1e-9)
    mask = (med - col) > thr
    if not mask.any():
        return None, None, False
    track_y = y_mm[mask]
    return float(track_y.min()), float(track_y.max()), True


def old_width_trace(Z_mm, x_mm, y_mm, x_centers, window_mm,
                     nan_frac_max=0.6, min_valid_cols=5, mad_k=3.0):
    out = np.full(len(x_centers), np.nan)
    for i, xc in enumerate(x_centers):
        col_mask = (x_mm >= xc - window_mm / 2) & (x_mm <= xc + window_mm / 2)
        col_indices = np.flatnonzero(col_mask)
        if col_indices.size == 0:
            continue
        block = Z_mm[:, col_indices]
        if float(np.isnan(block).sum() / block.size) > nan_frac_max:
            continue
        widths = []
        for idx in col_indices:
            left, right, ok = _old_column_track_boundary(Z_mm[:, idx], y_mm, mad_k=mad_k)
            if ok:
                widths.append(right - left)
        if len(widths) >= min_valid_cols:
            out[i] = float(np.mean(widths))
    return out


DEMO_TRACK = 8
height_dir = REPO / 'data' / 'raw' / 'height_maps'
height = load_wyko_asc(height_dir, DEMO_TRACK, crop_to_common=True)
Z_mm, _ = robust_plane_detrend(height['Z_mm'], height['x_actual_mm'], height['y_mm'])
x_actual_mm, y_mm = height['x_actual_mm'], height['y_mm']

x_centers = np.arange(40.0, 60.0, 0.2)
old_widths = old_width_trace(Z_mm, x_actual_mm, y_mm, x_centers, THERMAL_MM_PER_FRAME)
new_stats = new_extract_local_width_stats(Z_mm, x_actual_mm, y_mm, x_centers, THERMAL_MM_PER_FRAME)
new_widths = np.where(new_stats['valid'], new_stats['width_mean_mm'], np.nan)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(x_centers, old_widths, color='#888888', lw=1.2, marker='.', ms=3,
        label='old: per-column MAD boundary (pre-2026-07-16)')
ax.plot(x_centers, new_widths, color='#cc3333', lw=1.5, marker='.', ms=3,
        label='new: NaN-valley band (current)')
ax.set_xlabel('x (mm)')
ax.set_ylabel('width (mm)')
ax.set_title(f'Target extraction, old vs new — track {DEMO_TRACK}, x=40–60mm')
ax.legend(loc='upper right', fontsize=9)
fig.tight_layout()
fig.savefig(FIG_DIR / 'target_redefinition_before_after.png', dpi=150)
plt.show()

## TODO: remaining figures (not built yet)

- Discovery-narrative figures: CVAE fold-3 anomaly, RBF-vs-linear-kernel GP comparison
  (today's probe_waviness_gp* results) -- optional, can describe in text instead
- Registration-ambiguity illustration: centerline offset sign flip across dev tracks
  (+0.71 vs −0.77) -- optional, can describe in text instead